# Recalibrated AAE vs. t₀ plot with updating boxes

Place `lahm_issw_all_samples_recalibrated_grouped_2026-07-22.csv` in the same folder as this notebook. The pine-pollen, fullerene, and Rainier-dirt boxes are recalculated from the available CSV measurements each time the notebook runs.

In [ ]:
from pathlib import Path
import hashlib

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd


CSV_FILE = Path("lahm_issw_all_samples_recalibrated_grouped_2026-07-22.csv")
OUTPUT_FILE = Path("AAE_t0_recalibrated_updated_boxes.png")
LABEL_POINTS = False

T0_COLUMN = "t_0 value"
AAE_COLUMN = "AAE value"
LAHM_NAME_COLUMN = "LAHM file name"
ISSW_NAME_COLUMN = "ISSW file name"
GROUP_COLUMN = "Plot group"

# These established groups retain the same color every time the plot is run.
PREFERRED_COLORS = {
    "Mount Adams": "#1f77b4",
    "Mount Baker crater": "#ff7f0e",
    "Mount Baker summit": "#8c564b",
    "Mount Cook": "#17becf",
    "Other field samples": "#7f7f7f",
    "Pine pollen": "#9467bd",
    "Rainier Muir": "#2ca02c",
    "Rainier summit": "#bcbd22",
    "Rainier Site 1 surface": "#e377c2",
    "Rainier Site 1 subsurface": "#d62728",
    "Fullerene standards": "#4c4c4c",
    "Rainier dirt": "#a65628",
    "Other samples": "#6b7280",
}


def infer_plot_group(row):
    """Provide a fallback group when the CSV group cell is blank."""
    lahm = str(row.get(LAHM_NAME_COLUMN, "")).strip().lower()
    issw = str(row.get(ISSW_NAME_COLUMN, "")).strip().lower()
    combined = f"{lahm} {issw}"
    issw_stem = Path(issw).stem

    if issw_stem in {"1a", "1b"}:
        return "Rainier Site 1 surface"
    if issw_stem in {"2a", "3a", "4a", "5a", "6"}:
        return "Rainier Site 1 subsurface"
    if "pollen" in combined or "pp2026" in combined:
        return "Pine pollen"
    if "baker" in combined and "crater" in combined:
        return "Mount Baker crater"
    if "baker" in combined and "summit" in combined:
        return "Mount Baker summit"
    if "adams" in combined:
        return "Mount Adams"
    if "muir" in combined:
        return "Rainier Muir"
    if "rainier" in combined and "summit" in combined:
        return "Rainier summit"
    if "rainier dirt" in combined or "dirt str" in combined:
        return "Rainier dirt"
    if "mtcook" in combined or "mt cook" in combined:
        return "Mount Cook"
    if issw_stem.removesuffix(" ug").replace(" ", "").isdigit() or "fullerene" in combined:
        return "Fullerene standards"
    if any(term in combined for term in [
        "huann", "sea_ice", "greenland", "saguan", "saguyan", "sanguyan", "poi",
    ]):
        return "Other field samples"
    return "Other samples"


def color_for_group(group):
    """Return a stable color, including for group names added in the future."""
    if group in PREFERRED_COLORS:
        return PREFERRED_COLORS[group]

    # A stable hash means adding another group does not change existing colors.
    palette = plt.get_cmap("tab20").colors
    digest = hashlib.sha256(group.encode("utf-8")).digest()
    return palette[int.from_bytes(digest[:2], "big") % len(palette)]


def padded_bounds(values, fallback, minimum_padding, padding_fraction=0.08):
    """Return a padded data range, or a documented fallback if data is absent."""
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return fallback, False

    lower = float(values.min())
    upper = float(values.max())
    padding = max(minimum_padding, (upper - lower) * padding_fraction)
    return (lower - padding, upper + padding), True


def add_sample_range_box(
    ax,
    group,
    label,
    fallback_x,
    fallback_y,
):
    """Add one existing sample box using the newest available CSV ranges.

    The box shows the observed min-to-max range plus a small visual margin. It
    is not a confidence interval. Each axis updates independently, allowing the
    available AAE range to update even when matching t_0 values are still NaN.
    """
    group_data = data[data[GROUP_COLUMN] == group]
    x_bounds, x_updated = padded_bounds(
        group_data[T0_COLUMN], fallback_x, minimum_padding=0.05
    )
    y_bounds, y_updated = padded_bounds(
        group_data[AAE_COLUMN], fallback_y, minimum_padding=0.05
    )

    x_min, x_max = x_bounds
    y_min, y_max = y_bounds
    color = color_for_group(group)
    if x_updated and y_updated:
        box_label = f"{label}\n(observed range)"
    elif x_updated:
        box_label = f"{label}\n(t_0 range; AAE pending)"
    elif y_updated:
        box_label = f"{label}\n(AAE range; t_0 pending)"
    else:
        box_label = f"{label}\n(previous range)"
    ax.add_patch(
        Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            facecolor=color,
            edgecolor=color,
            linewidth=1.4,
            alpha=0.16,
            zorder=1,
        )
    )
    ax.text(
        (x_min + x_max) / 2,
        (y_min + y_max) / 2,
        box_label,
        ha="center",
        va="center",
        fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.55, "pad": 1.5},
        zorder=2,
    )
    print(
        f"{box_label.replace(chr(10), ' ')} box: t_0={x_min:.3f} to {x_max:.3f}; "
        f"AAE={y_min:.3f} to {y_max:.3f}; "
        f"updated axes: t_0={x_updated}, AAE={y_updated}"
    )


data = pd.read_csv(
    CSV_FILE,
    na_values=["NaN", "nan", "NAN", ""],
    keep_default_na=True,
)
data.columns = data.columns.str.strip()

required_columns = {
    LAHM_NAME_COLUMN,
    T0_COLUMN,
    ISSW_NAME_COLUMN,
    AAE_COLUMN,
}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(
        "The CSV is missing these required columns: "
        + ", ".join(sorted(missing_columns))
    )

for column in [LAHM_NAME_COLUMN, ISSW_NAME_COLUMN]:
    data[column] = data[column].astype("string").str.strip().replace("", pd.NA)

data[T0_COLUMN] = pd.to_numeric(data[T0_COLUMN], errors="coerce")
data[AAE_COLUMN] = pd.to_numeric(data[AAE_COLUMN], errors="coerce")
data["Sample label"] = data[ISSW_NAME_COLUMN].fillna(data[LAHM_NAME_COLUMN])

if GROUP_COLUMN not in data.columns:
    data[GROUP_COLUMN] = pd.NA
data[GROUP_COLUMN] = data[GROUP_COLUMN].astype("string").str.strip().replace("", pd.NA)
data[GROUP_COLUMN] = data.apply(
    lambda row: row[GROUP_COLUMN]
    if pd.notna(row[GROUP_COLUMN])
    else infer_plot_group(row),
    axis=1,
)

complete = data.dropna(subset=[T0_COLUMN, AAE_COLUMN]).copy()
incomplete = data[data[[T0_COLUMN, AAE_COLUMN]].isna().any(axis=1)].copy()
if complete.empty:
    raise ValueError("No CSV rows contain both a t_0 value and an AAE value.")

print(f"Loaded {len(data)} CSV rows.")
print(f"Plotting {len(complete)} complete rows in {complete[GROUP_COLUMN].nunique()} groups.")
print(f"Skipping {len(incomplete)} rows missing t_0 or AAE.")
print("\nPlotted groups:")
print(complete[GROUP_COLUMN].value_counts().sort_index().to_string())


# Retain the original view but expand the axes when new data falls outside it.
x_range = complete[T0_COLUMN].max() - complete[T0_COLUMN].min()
y_range = complete[AAE_COLUMN].max() - complete[AAE_COLUMN].min()
x_padding = max(0.2, 0.05 * x_range)
y_padding = max(0.2, 0.05 * y_range)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(
    min(3.5, complete[T0_COLUMN].min() - x_padding),
    max(7.5, complete[T0_COLUMN].max() + x_padding),
)
ax.set_ylim(
    min(-1.2, complete[AAE_COLUMN].min() - y_padding),
    max(3.2, complete[AAE_COLUMN].max() + y_padding),
)
ax.set_xlabel(r"$t_0$ (s)")
ax.set_ylabel("Absorption Ångström exponent (AAE)")
ax.grid(True, alpha=0.25)
ax.set_axisbelow(True)

# Keep the igneous-rock box fixed because it is a literature reference rather
# than a range calculated from this CSV.
ax.add_patch(
    Rectangle(
        (3.95, 1.5),
        2.5,
        1.5,
        facecolor="red",
        edgecolor="red",
        linewidth=1.4,
        alpha=0.16,
        zorder=1,
    )
)
ax.text(
    5.2,
    2.25,
    "igneous rock\n(literature)",
    ha="center",
    va="center",
    fontsize=9,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.55, "pad": 1.5},
    zorder=2,
)

# Update the three sample-derived boxes from the newest CSV. These are the same
# boxes as before; only their bounds and colors are data-driven.
add_sample_range_box(
    ax,
    group="Rainier dirt",
    label="Rainier dirt",
    fallback_x=(4.9, 6.9),
    fallback_y=(0.0, 0.4),
)
add_sample_range_box(
    ax,
    group="Pine pollen",
    label="pine pollen",
    fallback_x=(6.3, 6.7),
    fallback_y=(1.5, 2.1),
)
add_sample_range_box(
    ax,
    group="Fullerene standards",
    label="fullerene standards",
    fallback_x=(5.1, 5.7),
    fallback_y=(-1.0, 0.0),
)


# Every group is discovered from the CSV at run time. New rows in an existing
# group reuse its color; a new group name receives a deterministic new color.
for group, group_data in complete.groupby(GROUP_COLUMN, sort=True):
    ax.scatter(
        group_data[T0_COLUMN],
        group_data[AAE_COLUMN],
        marker="o",
        s=58,
        color=color_for_group(group),
        edgecolor="white",
        linewidth=0.6,
        label=group,
        zorder=3,
    )

    if LABEL_POINTS:
        for _, row in group_data.iterrows():
            ax.annotate(
                row["Sample label"],
                (row[T0_COLUMN], row[AAE_COLUMN]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=7,
            )

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    fontsize=8,
    frameon=True,
    borderaxespad=0,
)
fig.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")
plt.show()
